# Enterprise GCP Data Stack Architecture: Supply Chain & Inventory Optimization

**Project type:** Self-directed portfolio project aligned to the responsibilities described in the Google Data Specialist job posting (R00336594). Not client work; not affiliated with or endorsed by Accenture or Google.

**Project ID:** `driiiportfolio`

**Dataset:** `supply_chain_optimization`

**Execution Environment:** GCP Free-Tier (2026 Limits Constrained)

---

## Phase 1: Technical Discovery, Scenario, & Business Problem

### 1. Enterprise Client Scenario

The scenario models a fictional multi-region retail enterprise operating across 12 distribution centers and over 400 retail storefronts. The generated implementation intentionally uses 5 distribution centers; the 12-center figure is a scenario assumption, not a generated-data measurement. The legacy architecture relies on an on-premises transactional database coupled with batch-processed file transfers.

For this self-directed portfolio exercise, a fictional enterprise is assumed to engage a Google Cloud delivery team. No real client engagement is claimed.

```
+-----------------------------------------------------------------------------------+
|                            LEGACY INFRASTRUCTURE                                  |
|                                                                                   |
|  +--------------------+      +--------------------+      +---------------------+  |
|  | On-Premises ERP    | ---> | Batch File Transfer| ---> | Monolithic Relational|  |
|  | Transaction Logs   |      | (Nightly FTP/CSV)  |      | Database (Data Silo)|  |
|  +--------------------+      +--------------------+      +---------------------+  |
+-------------------------------------------------------------------------|---------+
                                                                          |
                                 GCP MODERNIZATION ROADMAP               |
                                                                          v
+-----------------------------------------------------------------------------------+
|                        TARGET GOOGLE CLOUD ARCHITECTURE                           |
|                                                                                   |
|  +--------------------+      +--------------------+      +---------------------+  |
|  | Colab Ingestion    | ---> | BigQuery Warehouse | ---> | Vertex AI / Colab   |  |
|  | Pipeline (Python)  |      | Datasets & Views   |      | Statistical Engine  |  |
|  +--------------------+      +--------------------+      +---------------------+  |
|                                                                     |             |
|                                                                     v             |
|                                                          +---------------------+  |
|                                                          | Looker Studio       |  |
|                                                          | Control Tower       |  |
|                                                          +---------------------+  |
+-----------------------------------------------------------------------------------+

```

### 2. Business Problem & Operational Bottlenecks

* **Data Silo Constraints:** Transactional sales, warehouse log processing, and supplier fulfillment feeds reside in isolated databases. Cross-domain queries take hours or fail due to resource contention.
* **Query Latency & Stale Telemetry:** Nightly ETL batch runs mean warehouse leads operate on data that is 12–24 hours old. Inventory balancing decisions are reactive rather than predictive.
* **Fulfillment Bottlenecks & High Carrying Costs:** Inability to accurately estimate lead times across distinct fulfillment routes forces regional warehouses to maintain up to 28% excess safety stock, while fast-moving SKUs experience localized stockout rates exceeding 14%.

### 3. Engineering Roadmap

1. **Data Ingestion:** Generate seeded synthetic data in Python and batch-load normalized tables into BigQuery. Streaming is a billing-gated production pattern, not the default sandbox path.
2. **Data Warehousing & Transformation Layer:** Author standard GoogleSQL transformations in BigQuery to sanitize, standardize timestamp types, build star-schema data models, and construct analytical views optimized for low-scan queries.
3. **Statistical Modeling & AI Diagnostics:** Pull transformed datasets from BigQuery into a statistical engine running SciPy and Statsmodels. Execute ANOVA tests on warehouse processing durations and Chi-Square tests on stockout frequencies.
4. **Visual Analytics & Business Intelligence:** Build and validate a Looker Studio Control Tower specification. The live report is an external deliverable and is not claimed as published from this notebook.

---

## Phase 2 & Phase 3: Proof-of-Value, Core Engineering, & Data Pipelines

### 1. Data Flow Blueprint & Methodology Justification

$$\text{Synthetic Data Generation} \longrightarrow \text{BigQuery Warehouse Transformations} \longrightarrow \text{Statistical Analysis Matrix} \longrightarrow \text{Looker Studio Delivery}$$

#### Methodology Justification

| Analysis Technique | Operational Bottleneck Addressed | Alternative Considered | Selection Rationale |
| --- | --- | --- | --- |
| **One-Way ANOVA (Analysis of Variance)** | Variable fulfillment delays across different distribution centers. | Pairwise Student’s t-tests across centers. | Multiple t-tests multiply Type I error rates. ANOVA tests equal population means across all distribution centers simultaneously without inflating alpha inflation risks. |
| **Chi-Square Test of Independence** | Unclear categorical relationship between warehouse locations and stockout frequency. | Logistic Regression Modeling. | Chi-Square directly evaluates independence between nominal variables (Location vs. Stock Status) without parametric assumptions, ideal for operational categorical diagnostics. |
| **Ordinary Least Squares (OLS) Multiple Regression** | Unpredictable reorder lead times causing safety stock over-allocation. | Simple Moving Averages (SMA). | SMA fails to isolate independent covariate drivers (carrier, warehouse, planned lead time). OLS quantifies specific variable contributions to lead-time variations. |

---

### 2. Google Colab Synthetic Data Ingestion Pipeline

The following Python code generates synthetic relational supply chain datasets, applies standard typing, and programmatically provisions BigQuery resources under the `driiiportfolio` GCP project.

In [ ]:
import os
import uuid
import random
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
from google.cloud import bigquery
from google.api_core.exceptions import GoogleAPIError

# Configuration Parameters
PROJECT_ID = "driiiportfolio"
DATASET_ID = "supply_chain_optimization"
LOCATION = "US"
NUM_WAREHOUSES = 5
NUM_PRODUCTS = 20
NUM_TRANSACTIONS = 1500
NUM_LOGS = 2500

def initialize_bigquery_client(project_id: str) -> bigquery.Client:
    """Initializes and returns a Google BigQuery Client instance."""
    try:
        client = bigquery.Client(project=project_id)
        print(f"[INFO] BigQuery client initialized successfully for project: {project_id}")
        return client
    except Exception as e:
        print(f"[ERROR] Failed to initialize BigQuery client: {e}")
        raise e

def create_bigquery_dataset(client: bigquery.Client, dataset_id: str, location: str) -> str:
    """Creates the target BigQuery dataset if it does not already exist."""
    dataset_ref = bigquery.DatasetReference(client.project, dataset_id)
    dataset = bigquery.Dataset(dataset_ref)
    dataset.location = location
    try:
        dataset = client.create_dataset(dataset, exists_ok=True)
        full_dataset_id = f"{client.project}.{dataset.dataset_id}"
        print(f"[INFO] Dataset {full_dataset_id} verified/created in location {location}.")
        return full_dataset_id
    except GoogleAPIError as e:
        print(f"[ERROR] Failed to create dataset {dataset_id}: {e}")
        raise e

def generate_synthetic_data():
    """Generates synthetic relational dataframes modeling supply chain operations."""
    print("[INFO] Starting synthetic enterprise dataset generation...")
    np.random.seed(42)
    random.seed(42)

    # 1. Warehouses Entity
    warehouses_data = []
    cities = ["Seattle", "Dallas", "Chicago", "Atlanta", "New York"]
    states = ["WA", "TX", "IL", "GA", "NY"]
    for i in range(NUM_WAREHOUSES):
        warehouses_data.append({
            "warehouse_id": f"WH-{100 + i}",
            "warehouse_name": f"Regional Distribution Center {cities[i]}",
            "city": cities[i],
            "state": states[i],
            "capacity_units": int(np.random.choice([50000, 75000, 100000, 150000])),
            "operating_cost_daily": round(float(np.random.uniform(2500.0, 6000.0)), 2)
        })
    df_warehouses = pd.DataFrame(warehouses_data)

    # 2. Inventory Transactions Entity
    categories = ["Electronics", "Apparel", "Home & Kitchen", "Automotive"]
    products = [f"SKU-{2000 + i}" for i in range(NUM_PRODUCTS)]

    transactions_data = []
    start_date = datetime(2026, 1, 1)

    for _ in range(NUM_TRANSACTIONS):
        tx_date = start_date + timedelta(
            days=random.randint(0, 180),
            hours=random.randint(0, 23),
            minutes=random.randint(0, 59)
        )
        tx_type = np.random.choice(["RESTOCK", "OUTBOUND", "ADJUSTMENT"], p=[0.35, 0.55, 0.10])
        qty = int(np.random.randint(5, 500)) if tx_type != "ADJUSTMENT" else int(np.random.randint(-20, 20))

        transactions_data.append({
            "transaction_id": str(uuid.uuid4()),
            "warehouse_id": f"WH-{100 + random.randint(0, NUM_WAREHOUSES - 1)}",
            "sku": random.choice(products),
            "category": random.choice(categories),
            "transaction_type": tx_type,
            "quantity": qty,
            "unit_cost": round(float(np.random.uniform(10.0, 250.0)), 2),
            "transaction_timestamp": tx_date.strftime("%Y-%m-%d %H:%M:%S")
        })
    df_transactions = pd.DataFrame(transactions_data)

    # 3. Warehouse Logistics & Fulfillment Logs Entity
    logs_data = []
    carriers = ["FedEx Freight", "UPS Supply Chain", "DHL Express", "JB Hunt"]
    stock_statuses = ["IN_STOCK", "LOW_STOCK", "STOCKOUT"]

    for _ in range(NUM_LOGS):
        dispatch_time = start_date + timedelta(
            days=random.randint(0, 180),
            hours=random.randint(0, 23),
            minutes=random.randint(0, 59)
        )
        planned_lead_days = random.randint(2, 7)
        # Delay is intentionally independent of carrier and warehouse.
        actual_lead_days = planned_lead_days + int(np.random.choice([0, 1, 2, 3, 5], p=[0.5, 0.25, 0.15, 0.07, 0.03]))
        delivery_time = dispatch_time + timedelta(days=actual_lead_days)

        logs_data.append({
            "log_id": f"LOG-{uuid.uuid4().hex[:8].upper()}",
            "warehouse_id": f"WH-{100 + random.randint(0, NUM_WAREHOUSES - 1)}",
            "sku": random.choice(products),
            "carrier": random.choice(carriers),
            "planned_lead_time_days": planned_lead_days,
            "actual_lead_time_days": actual_lead_days,
            "stock_status": np.random.choice(stock_statuses, p=[0.70, 0.20, 0.10]),
            "dispatch_timestamp": dispatch_time.strftime("%Y-%m-%d %H:%M:%S"),
            "delivery_timestamp": delivery_time.strftime("%Y-%m-%d %H:%M:%S")
        })
    df_logs = pd.DataFrame(logs_data)

    print("[INFO] Synthetic data generation complete.")
    return df_warehouses, df_transactions, df_logs

def save_and_load_to_bigquery(client: bigquery.Client, dataset_id: str, tables_dict: dict):
    """Saves DataFrames to local CSV files and loads them into BigQuery tables."""
    os.makedirs("data_export", exist_ok=True)

    for table_name, df in tables_dict.items():
        csv_path = os.path.join("data_export", f"{table_name}.csv")
        df.to_csv(csv_path, index=False)
        print(f"[INFO] Saved local CSV: {csv_path} ({len(df)} rows)")

        table_ref = client.dataset(dataset_id).table(table_name)
        job_config = bigquery.LoadJobConfig(
            source_format=bigquery.SourceFormat.CSV,
            skip_leading_rows=1,
            autodetect=True,
            write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
        )

        with open(csv_path, "rb") as source_file:
            load_job = client.load_table_from_file(
                source_file,
                table_ref,
                job_config=job_config
            )

        print(f"[INFO] Starting BigQuery load job for table: {table_name}...")
        load_job.result()  # Wait for job execution to complete

        destination_table = client.get_table(table_ref)
        print(f"[SUCCESS] Loaded {destination_table.num_rows} rows into {PROJECT_ID}.{dataset_id}.{table_name}")

def main():
    try:
        from google.colab import auth
        auth.authenticate_user()
    except ImportError:
        print("[INFO] Non-Colab execution: using existing Google Application Default Credentials.")
    client = initialize_bigquery_client(PROJECT_ID)
    create_bigquery_dataset(client, DATASET_ID, LOCATION)

    df_warehouses, df_transactions, df_logs = generate_synthetic_data()

    tables_to_upload = {
        "dim_warehouses": df_warehouses,
        "fact_inventory_transactions": df_transactions,
        "fact_warehouse_logs": df_logs
    }

    save_and_load_to_bigquery(client, DATASET_ID, tables_to_upload)
    print("\n[COMPLETE] Phase 2 Data Ingestion Pipeline executed successfully.")

if __name__ == "__main__":
    main()

---

### 3. BigQuery SQL Data Warehousing Scripts

> **Sandbox design:** staging tables are clustered but not date-partitioned. BigQuery Sandbox tables, views and partitions automatically expire after 60 days, while streaming and DML are unavailable in the sandbox. The project therefore uses batch load jobs and `CREATE OR REPLACE`/CTAS patterns.

**Execution order:** raw tables → staging tables → reporting views → reconciliation → statistical analysis → ML → quality/governance → performance.

**Primary Looker Studio sources:** `vw_shipment_detail` and `vw_inventory_daily`.

In [ ]:
from google.cloud import bigquery
from google.colab import auth

# Authenticate user for Colab environment if not already done
auth.authenticate_user()

client = bigquery.Client(project=PROJECT_ID)

def execute_bq_query(client: bigquery.Client, query: str, query_name: str = "BigQuery Query"):
    """Executes a BigQuery SQL query and handles potential errors."""
    try:
        query_job = client.query(query)
        query_job.result()  # Waits for the job to complete
        print(f"[SUCCESS] {query_name} executed successfully.")
    except Exception as e:
        print(f"[ERROR] Failed to execute {query_name}: {e}")
        # Re-raise the exception to stop execution if a critical error occurs
        raise e


In [ ]:
# SQL for stg_inventory_transactions
sql_stg_inventory_transactions = """
DROP TABLE IF EXISTS `driiiportfolio.supply_chain_optimization.stg_inventory_transactions`;
CREATE TABLE `driiiportfolio.supply_chain_optimization.stg_inventory_transactions`
CLUSTER BY warehouse_id, category AS
SELECT
  CAST(transaction_id AS STRING) AS transaction_id,
  UPPER(TRIM(warehouse_id)) AS warehouse_id,
  UPPER(TRIM(sku)) AS sku,
  INITCAP(TRIM(category)) AS category,
  UPPER(TRIM(transaction_type)) AS transaction_type,
  CAST(quantity AS INT64) AS quantity,
  ROUND(CAST(unit_cost AS NUMERIC), 2) AS unit_cost,
  ROUND(CAST(quantity AS NUMERIC) * CAST(unit_cost AS NUMERIC), 2) AS total_transaction_value,
  transaction_timestamp
FROM
  `driiiportfolio.supply_chain_optimization.fact_inventory_transactions`
WHERE
  transaction_id IS NOT NULL;
"""
execute_bq_query(client, sql_stg_inventory_transactions, "stg_inventory_transactions Table Creation")

# SQL for stg_warehouse_logs
sql_stg_warehouse_logs = """
DROP TABLE IF EXISTS `driiiportfolio.supply_chain_optimization.stg_warehouse_logs`;
CREATE TABLE `driiiportfolio.supply_chain_optimization.stg_warehouse_logs`
CLUSTER BY warehouse_id, carrier AS
SELECT
  CAST(log_id AS STRING) AS log_id,
  UPPER(TRIM(warehouse_id)) AS warehouse_id,
  UPPER(TRIM(sku)) AS sku,
  TRIM(carrier) AS carrier,
  CAST(planned_lead_time_days AS INT64) AS planned_lead_time_days,
  CAST(actual_lead_time_days AS INT64) AS actual_lead_time_days,
  (CAST(actual_lead_time_days AS INT64) - CAST(planned_lead_time_days AS INT64)) AS lead_time_variance_days,
  UPPER(TRIM(stock_status)) AS stock_status,
  dispatch_timestamp,
  delivery_timestamp
FROM
  `driiiportfolio.supply_chain_optimization.fact_warehouse_logs`
WHERE
  log_id IS NOT NULL;
"""
execute_bq_query(client, sql_stg_warehouse_logs, "stg_warehouse_logs Table Creation")


In [ ]:
# SQL for vw_inventory_health
sql_vw_inventory_health = """
CREATE OR REPLACE VIEW `driiiportfolio.supply_chain_optimization.vw_inventory_health` AS
SELECT
  t.warehouse_id,
  w.warehouse_name,
  w.city,
  w.state,
  t.category,
  COUNT(DISTINCT t.transaction_id) AS total_transactions,
  SUM(CASE WHEN t.transaction_type = 'RESTOCK' THEN t.quantity ELSE 0 END) AS units_restocked,
  SUM(CASE WHEN t.transaction_type = 'OUTBOUND' THEN t.quantity ELSE 0 END) AS units_dispatched,
  SUM(CASE WHEN t.transaction_type = 'ADJUSTMENT' THEN t.quantity ELSE 0 END) AS units_adjusted,
  ROUND(SUM(t.total_transaction_value), 2) AS total_gross_value,
  ROUND(AVG(t.unit_cost), 2) AS avg_unit_cost,
  CURRENT_TIMESTAMP() AS view_refreshed_at
FROM
  `driiiportfolio.supply_chain_optimization.stg_inventory_transactions` t
LEFT JOIN
  `driiiportfolio.supply_chain_optimization.dim_warehouses` w
ON
  t.warehouse_id = w.warehouse_id
GROUP BY
  1, 2, 3, 4, 5;
"""
execute_bq_query(client, sql_vw_inventory_health, "vw_inventory_health View Creation")

# SQL for vw_warehouse_performance
sql_vw_warehouse_performance = """
CREATE OR REPLACE VIEW `driiiportfolio.supply_chain_optimization.vw_warehouse_performance` AS
SELECT
  l.warehouse_id,
  w.warehouse_name,
  l.carrier,
  COUNT(l.log_id) AS total_shipments,
  ROUND(AVG(l.planned_lead_time_days), 2) AS avg_planned_lead_days,
  ROUND(AVG(l.actual_lead_time_days), 2) AS avg_actual_lead_days,
  ROUND(AVG(l.lead_time_variance_days), 2) AS avg_lead_time_delay_days,
  COUNTIF(l.lead_time_variance_days > 0) AS delayed_shipment_count,
  ROUND(SAFE_DIVIDE(COUNTIF(l.lead_time_variance_days > 0), COUNT(l.log_id)) * 100, 2) AS delay_rate_percentage,
  COUNTIF(l.stock_status = 'STOCKOUT') AS stockout_incidents,
  COUNTIF(l.stock_status = 'LOW_STOCK') AS low_stock_warnings,
  ROUND(SAFE_DIVIDE(COUNTIF(l.stock_status = 'STOCKOUT'), COUNT(l.log_id)) * 100, 2) AS stockout_rate_percentage
FROM
  `driiiportfolio.supply_chain_optimization.stg_warehouse_logs` l
LEFT JOIN
  `driiiportfolio.supply_chain_optimization.dim_warehouses` w
ON
  l.warehouse_id = w.warehouse_id
GROUP BY
  1, 2, 3;
"""
execute_bq_query(client, sql_vw_warehouse_performance, "vw_warehouse_performance View Creation")


# Dashboard source: one row per shipment
sql_vw_shipment_detail = """
CREATE OR REPLACE VIEW `driiiportfolio.supply_chain_optimization.vw_shipment_detail` AS
SELECT l.log_id,l.warehouse_id,w.warehouse_name,w.city,w.state,l.sku,l.carrier,
       l.planned_lead_time_days,l.actual_lead_time_days,l.lead_time_variance_days,l.stock_status,
       DATE(l.dispatch_timestamp) AS dispatch_date,l.dispatch_timestamp,l.delivery_timestamp,
       IF(l.lead_time_variance_days<=0,1,0) AS is_on_time,
       IF(l.lead_time_variance_days>0,1,0) AS is_delayed,
       IF(l.stock_status='STOCKOUT',1,0) AS is_stockout,
       IF(l.stock_status='LOW_STOCK',1,0) AS is_low_stock
FROM `driiiportfolio.supply_chain_optimization.stg_warehouse_logs` l
LEFT JOIN `driiiportfolio.supply_chain_optimization.dim_warehouses` w USING (warehouse_id)
"""
execute_bq_query(client, sql_vw_shipment_detail, "vw_shipment_detail View Creation")

# Dashboard source: daily inventory grain
sql_vw_inventory_daily = """
CREATE OR REPLACE VIEW `driiiportfolio.supply_chain_optimization.vw_inventory_daily` AS
SELECT DATE(t.transaction_timestamp) AS transaction_date,t.warehouse_id,w.warehouse_name,w.city,w.state,
       t.category,t.transaction_type,COUNT(*) AS transactions,ROUND(SUM(t.total_transaction_value),2) AS gross_value
FROM `driiiportfolio.supply_chain_optimization.stg_inventory_transactions` t
LEFT JOIN `driiiportfolio.supply_chain_optimization.dim_warehouses` w USING (warehouse_id)
GROUP BY 1,2,3,4,5,6,7
"""
execute_bq_query(client, sql_vw_inventory_daily, "vw_inventory_daily View Creation")

In [ ]:
from google.cloud import bigquery
import pandas as pd

client = bigquery.Client(project=PROJECT_ID)

def fetch_and_display_table_head(client: bigquery.Client, table_name: str, dataset_id: str = DATASET_ID, num_rows: int = 5):
    """Fetches the first few rows of a BigQuery table/view and displays them."""
    full_table_name = f"`{PROJECT_ID}.{dataset_id}.{table_name}`"
    query = f"SELECT * FROM {full_table_name} LIMIT {num_rows}"
    print(f"\n[INFO] Querying {full_table_name}...")
    try:
        df = client.query(query).to_dataframe()
        print(f"[SUCCESS] Retrieved {len(df)} rows from {table_name}.")
        display(df)
    except Exception as e:
        print(f"[ERROR] Failed to query {full_table_name}: {e}")

# Verify stg_inventory_transactions
fetch_and_display_table_head(client, "stg_inventory_transactions")

# Verify stg_warehouse_logs
fetch_and_display_table_head(client, "stg_warehouse_logs")
fetch_and_display_table_head(client, "vw_shipment_detail")
fetch_and_display_table_head(client, "vw_inventory_daily")

# Verify vw_inventory_health
fetch_and_display_table_head(client, "vw_inventory_health")

# Verify vw_warehouse_performance
fetch_and_display_table_head(client, "vw_warehouse_performance")


# Reconciliation: staging must match raw facts; views must roll up to staging totals
recon = f"""
SELECT 'stg_inventory_transactions' AS obj, COUNT(*) AS n, 1500 AS expected FROM `{PROJECT_ID}.{DATASET_ID}.stg_inventory_transactions`
UNION ALL SELECT 'stg_warehouse_logs', COUNT(*), 2500 FROM `{PROJECT_ID}.{DATASET_ID}.stg_warehouse_logs`
UNION ALL SELECT 'vw_inventory_health (sum tx)', SUM(total_transactions), 1500 FROM `{PROJECT_ID}.{DATASET_ID}.vw_inventory_health`
UNION ALL SELECT 'vw_warehouse_performance (sum shipments)', SUM(total_shipments), 2500 FROM `{PROJECT_ID}.{DATASET_ID}.vw_warehouse_performance`
"""
df_chk = client.query(recon).to_dataframe()
df_chk["PASS"] = df_chk["n"] == df_chk["expected"]
display(df_chk)
assert df_chk["PASS"].all(), "Row-count reconciliation failed"


recon = f"""SELECT 'shipment detail rows' check_name, COUNT(*) actual, 2500 expected FROM `{PROJECT_ID}.{DATASET_ID}.vw_shipment_detail` UNION ALL SELECT 'inventory daily transactions', SUM(transactions), 1500 FROM `{PROJECT_ID}.{DATASET_ID}.vw_inventory_daily`"""
df_dashboard_recon=client.query(recon).to_dataframe()
df_dashboard_recon["PASS"]=df_dashboard_recon.actual==df_dashboard_recon.expected
display(df_dashboard_recon)
assert df_dashboard_recon.PASS.all()

In [ ]:
"""
Filename: scripts/statistical_analysis.py
Description: Pulls transformed BigQuery views, executes advanced statistical analysis
             (ANOVA, Chi-Square, OLS Regression), and outputs diagnostic matrices.
Author: Portfolio project author
Target Environment: Google Colab / Python 3.10+ / GCP Free Tier (2026)
"""

import sys
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from google.cloud import bigquery

# Project Configuration
PROJECT_ID = "driiiportfolio"
DATASET_ID = "supply_chain_optimization"

def fetch_bq_data(query: str, project_id: str) -> pd.DataFrame:
    """Executes SQL query against BigQuery and returns a Pandas DataFrame."""
    client = bigquery.Client(project=project_id)
    try:
        df = client.query(query).to_dataframe()
        print(f"[INFO] Successfully queried {len(df)} records from BigQuery.")
        return df
    except Exception as e:
        print(f"[ERROR] BigQuery query execution failed: {e}")
        sys.exit(1)

def run_anova_warehouse_lead_times(df_logs: pd.DataFrame):
    """Performs One-Way ANOVA across distinct warehouse locations on actual lead times."""
    print("\n" + "="*80)
    print("1. STATISTICAL EVALUATION: ONE-WAY ANOVA (Warehouse Lead Time Variance)")
    print("="*80)

    # Group lead times by warehouse
    groups = [group["actual_lead_time_days"].values for _, group in df_logs.groupby("warehouse_id")]

    f_stat, p_val = stats.f_oneway(*groups)

    print(f"Calculated F-Statistic: {f_stat:.4f}")
    print(f"Calculated P-Value:     {p_val:.4e}")

    if p_val < 0.05:
        print("[CONCLUSION] Reject Null Hypothesis (H0). Statistically significant differences exist")
        print("             in mean actual lead times across distribution centers (p < 0.05).")
    else:
        print("[CONCLUSION] Fail to Reject H0. No statistically significant difference in lead times")
        print("             detected across distribution centers.")

def run_chi_square_stockouts(df_logs: pd.DataFrame):
    """Executes Chi-Square Test of Independence between Warehouse and Stock Status."""
    print("\n" + "="*80)
    print("2. STATISTICAL EVALUATION: CHI-SQUARE TEST OF INDEPENDENCE (Stockout Risk)")
    print("="*80)

    contingency_table = pd.crosstab(df_logs["warehouse_id"], df_logs["stock_status"])
    print("Contingency Table (Warehouse vs Stock Status):")
    print(contingency_table)

    chi2, p_val, dof, expected = stats.chi2_contingency(contingency_table)

    print(f"\nChi-Square Statistic: {chi2:.4f}")
    print(f"Degrees of Freedom:  {dof}")
    print(f"Calculated P-Value:   {p_val:.4e}")

    if p_val < 0.05:
        print("[CONCLUSION] Reject H0. Stockout frequency is dependent on warehouse location (p < 0.05).")
    else:
        print("[CONCLUSION] Fail to Reject H0. Stockout status is independent of warehouse location.")

def run_ols_lead_time_regression(df_logs: pd.DataFrame):
    """Fits an Ordinary Least Squares (OLS) regression model to predict lead time delays."""
    print("\n" + "="*80)
    print("3. PREDICTIVE EVALUATION: OLS MULTIPLE REGRESSION MODEL")
    print("="*80)

    # Feature engineering for regression
    df_logs["lead_delay"] = df_logs["actual_lead_time_days"] - df_logs["planned_lead_time_days"]

    model = ols('lead_delay ~ C(carrier) + C(warehouse_id) + planned_lead_time_days', data=df_logs).fit()

    print(model.summary())

def main():
    # Query logs data directly from sanitized BigQuery table
    logs_query = f"""
        SELECT
            warehouse_id,
            carrier,
            planned_lead_time_days,
            actual_lead_time_days,
            stock_status
        FROM `{PROJECT_ID}.{DATASET_ID}.stg_warehouse_logs`
        ORDER BY log_id
    """

    df_logs = fetch_bq_data(logs_query, PROJECT_ID)

    # Run statistical engines
    run_anova_warehouse_lead_times(df_logs)
    run_chi_square_stockouts(df_logs)
    run_ols_lead_time_regression(df_logs)

if __name__ == "__main__":
    main()

### Analysis Results & Interpretation (validation run 2026-09-30, n = 2,500)

| Test | Statistic | p-value | Interpretation |
|---|---:|---:|---|
| One-way ANOVA (actual lead time by warehouse) | F = 0.174 | 0.952 | No statistically significant warehouse difference |
| Chi-Square (warehouse × stock status) | chi2 = 3.342, df = 8 | 0.911 | No evidence of dependence |
| OLS (delay ~ carrier + warehouse + planned lead) | R² = 0.002, F = 0.575 | 0.799 | No statistically significant model signal |

The synthetic generator intentionally draws warehouse, carrier, SKU and delay independently. These null results are expected and must not be presented as evidence about real logistics operations.

The statistical query is ordered by `log_id` for deterministic row order. Durbin–Watson is not used as a substantive finding because these observations are not a time series.

---
## Phase 2B: Requirements Traceability & Delivery Log
*(JD Section 3: requirements gathering, translating requirements into tasks, communicating blockers)*

The Phase 1 scenario is treated as the client brief. Each requirement is traced to a technical task, a deliverable in this notebook, and an acceptance check.

| # | Client requirement | Technical task | Deliverable | Acceptance check |
|---|---|---|---|---|
| R1 | Remove 12-24 hour stale telemetry | Near-real-time ingestion alongside batch loads | "Ingestion patterns" section (billing-gated) | Streaming pattern is billing-gated; default path is batch-loaded |
| R2 | Reduce stockouts and excess safety stock caused by unpredictable lead times | Lead-time variance and stockout KPIs; delay classifier | `vw_warehouse_performance`, `mdl_delay_classifier` | KPI totals reconcile to staging (2,500 shipments) |
| R3 | Break down silos between transactions, logs and warehouses | Star schema, staging layer, reporting views | `dim_*`, `fact_*`, `stg_*`, `vw_*` | 15 of 15 data-quality checks pass |
| R4 | Cut query latency and cost | Column pruning, clustering, views | "Performance tuning" section | Bytes scanned reported per query |
| R5 | Governance, auditability, least privilege | Data-quality checks, metadata, lineage, IAM review | "Governance" section | `dq_check_results` all PASS; IAM audit output |
| R6 | Executive visibility | Looker Studio Control Tower | Dashboard blueprint plus validation targets | Looker totals match BigQuery (247 stockouts, 52.1% fulfillment) |

### Delivery log: blockers, root causes, resolutions

| Issue | Root cause | Resolution |
|---|---|---|
| Staging tables returned 0 rows | BigQuery sandbox expires partitions after 60 days; data was dated Jan-Jun 2026 | Removed date partitioning, dropped and recreated tables, kept clustering |
| Views failed with 404 | Notebook cells were run out of order | Enforced run order; added a reconciliation assert |
| `PARSE_TIMESTAMP` error | Columns were already `TIMESTAMP` after autodetect | Removed the parse call |
| Executive summary contradicted results | Findings were written before the analysis ran | Rewrote findings to match ANOVA, Chi-Square and OLS output |
| DML and Gemini calls rejected | Sandbox requires billing for both | Used CTAS and load-job appends; flagged billing-gated cells |

---
## Phase 4B: ML Pipeline & MLOps with BigQuery ML
*(JD Section 2: ML models, pipelines, MLOps monitoring/retraining/deployment, model testing, validation and documentation)*

A logistic-regression classifier predicts whether a shipment is delayed (`lead_time_variance_days > 0`). The pipeline trains, evaluates against a majority-class baseline, applies a promotion gate, and logs every run to `ml_model_metrics`.

In [ ]:
from google.cloud import bigquery
import pandas as pd

sql_train_model = f"""
CREATE OR REPLACE MODEL `{PROJECT_ID}.{DATASET_ID}.mdl_delay_classifier`
OPTIONS (
  model_type = 'LOGISTIC_REG',
  input_label_cols = ['is_delayed'],
  data_split_method = 'RANDOM',
  data_split_eval_fraction = 0.25,
  auto_class_weights = FALSE
) AS
SELECT
  IF(lead_time_variance_days > 0, 1, 0) AS is_delayed,
  carrier,
  warehouse_id,
  planned_lead_time_days,
  EXTRACT(DAYOFWEEK FROM dispatch_timestamp) AS dispatch_dow
FROM `{PROJECT_ID}.{DATASET_ID}.stg_warehouse_logs`
"""
execute_bq_query(client, sql_train_model, "mdl_delay_classifier training")

In [ ]:
AUC_GATE = 0.60   # promotion gate: below this the model has no usable signal

eval_sql = f"""
SELECT accuracy, precision AS precision_score, recall AS recall_score, f1_score, roc_auc, log_loss
FROM ML.EVALUATE(MODEL `{PROJECT_ID}.{DATASET_ID}.mdl_delay_classifier`)
"""
base_sql = f"""
SELECT COUNT(*) AS training_rows,
       GREATEST(COUNTIF(lead_time_variance_days > 0), COUNTIF(lead_time_variance_days <= 0)) / COUNT(*) AS majority_baseline
FROM `{PROJECT_ID}.{DATASET_ID}.stg_warehouse_logs`
"""
ev = client.query(eval_sql).to_dataframe().iloc[0]
bs = client.query(base_sql).to_dataframe().iloc[0]
promoted = bool(ev["roc_auc"] >= AUC_GATE)

run_row = pd.DataFrame([{
    "run_at": pd.Timestamp.now(tz="UTC"),
    "model_name": "mdl_delay_classifier",
    "training_rows": int(bs["training_rows"]),
    "accuracy": float(ev["accuracy"]),
    "majority_baseline": float(bs["majority_baseline"]),
    "precision_score": float(ev["precision_score"]),
    "recall_score": float(ev["recall_score"]),
    "f1_score": float(ev["f1_score"]),
    "roc_auc": float(ev["roc_auc"]),
    "log_loss": float(ev["log_loss"]),
    "promoted": promoted,
    "note": "meets AUC gate" if promoted else f"below AUC gate {AUC_GATE:.2f}: not promoted",
}])

# Append via a load job (DML INSERT is blocked in the BigQuery sandbox)
load_job = client.load_table_from_dataframe(
    run_row,
    f"{PROJECT_ID}.{DATASET_ID}.ml_model_metrics",
    job_config=bigquery.LoadJobConfig(write_disposition="WRITE_APPEND"),
)
load_job.result()

print(f"ROC AUC {ev['roc_auc']:.3f} | accuracy {ev['accuracy']:.3f} vs majority baseline {bs['majority_baseline']:.3f}")
print("PROMOTED" if promoted else "NOT PROMOTED: no usable signal above the baseline")

display(client.query(f"""
SELECT run_at, model_name, training_rows, ROUND(accuracy,4) AS accuracy,
       ROUND(majority_baseline,4) AS majority_baseline, ROUND(roc_auc,4) AS roc_auc, promoted, note
FROM `{PROJECT_ID}.{DATASET_ID}.ml_model_metrics` ORDER BY run_at DESC
""").to_dataframe())

### Model card: `mdl_delay_classifier`

| Item | Detail |
|---|---|
| Purpose | Predict whether a shipment is delayed (`lead_time_variance_days > 0`) |
| Data | `stg_warehouse_logs`, 2,500 synthetic rows, 75/25 random train/eval split |
| Features | `carrier`, `warehouse_id`, `planned_lead_time_days`, dispatch day of week |
| Result (validation run 2026-09-30) | ROC AUC 0.527; accuracy 0.527 vs 0.521 majority baseline; precision and recall 0.0 (predicts the majority class) |
| Decision | **Not promoted** (AUC gate 0.60). This agrees with the ANOVA, Chi-Square and OLS results: the synthetic delays are independent of every feature |
| Monitoring | Each run is appended to `ml_model_metrics`; a falling AUC or accuracy against the baseline triggers review |
| Retraining | Re-run the two cells above on a schedule (BigQuery scheduled query, or Cloud Scheduler calling this notebook) |
| Deployment (not executed) | Register in Vertex AI Model Registry with the `model_registry = 'VERTEX_AI'` option at `CREATE MODEL` time, or export the model to Cloud Storage |
| Limitations | Synthetic data, no real signal. Metrics can shift slightly between runs because the split is random |

---
## Phase 4C: Gemini, Embeddings & RAG Experiment (billing required)
*(JD Section 2: Gemini foundation models, model APIs and embeddings, prompt engineering, RAG experimentation)*

> **Status: code provided, NOT executed.** The free-tier sandbox rejects Gemini calls ("requires billing to be enabled"). Set `RUN_BILLING_CELLS = True` only on a billing-enabled project, and verify the syntax against current BigQuery documentation.

One-time setup for the embedding model (run in Cloud Shell): create a Cloud resource connection with `bq mk --connection --location=US --connection_type=CLOUD_RESOURCE gemini_conn`, then grant its service account the **Vertex AI User** role.

In [ ]:
RUN_BILLING_CELLS = False   # flip to True only on a billing-enabled project
CONN = "gemini_conn"        # BigQuery Cloud resource connection (see setup note above)
T = f"{PROJECT_ID}.{DATASET_ID}"

if RUN_BILLING_CELLS:
    # 1) Prompt engineering: grounded operations notes (constrained, no invented causes)
    df_notes = client.query(f"""
    WITH kpis AS (
      SELECT warehouse_id, COUNT(*) AS shipments, COUNTIF(stock_status = 'STOCKOUT') AS stockouts,
             ROUND(AVG(lead_time_variance_days), 2) AS avg_delay_days
      FROM `{T}.stg_warehouse_logs` GROUP BY 1)
    SELECT warehouse_id,
           AI.GENERATE(CONCAT('In two sentences, write an operations note for warehouse ', warehouse_id,
             ' with ', CAST(shipments AS STRING), ' shipments, ', CAST(stockouts AS STRING),
             ' stockouts and an average delay of ', CAST(avg_delay_days AS STRING),
             ' days. Use only these numbers and do not invent causes.'),
             endpoint => 'gemini-2.5-flash').result AS ops_note
    FROM kpis ORDER BY warehouse_id""").to_dataframe()
    display(df_notes)

    # 2) RAG: embed KPI documents, retrieve with vector search, answer with Gemini grounded on the hits
    execute_bq_query(client, f"""
    CREATE OR REPLACE MODEL `{T}.mdl_embedding`
    REMOTE WITH CONNECTION `{PROJECT_ID}.us.{CONN}`
    OPTIONS (ENDPOINT = 'text-embedding-005')""", "embedding model")

    execute_bq_query(client, f"""
    CREATE OR REPLACE TABLE `{T}.kb_ops_docs` AS
    SELECT CONCAT(warehouse_id, '|', carrier) AS doc_id,
           CONCAT('Warehouse ', warehouse_id, ' with carrier ', carrier, ': ', CAST(COUNT(*) AS STRING),
                  ' shipments, ', CAST(COUNTIF(stock_status = 'STOCKOUT') AS STRING), ' stockouts, ',
                  CAST(ROUND(AVG(lead_time_variance_days), 2) AS STRING), ' average delay days.') AS content
    FROM `{T}.stg_warehouse_logs` GROUP BY warehouse_id, carrier""", "RAG documents")

    execute_bq_query(client, f"""
    CREATE OR REPLACE TABLE `{T}.kb_ops_embeddings` AS
    SELECT * FROM ML.GENERATE_EMBEDDING(MODEL `{T}.mdl_embedding`,
      (SELECT doc_id, content FROM `{T}.kb_ops_docs`), STRUCT(TRUE AS flatten_json_output))""", "document embeddings")

    question = "Which warehouse and carrier combinations have the most stockouts?"
    df_rag = client.query(f"""
    WITH hits AS (
      SELECT base.content AS content
      FROM VECTOR_SEARCH(TABLE `{T}.kb_ops_embeddings`, 'ml_generate_embedding_result',
        (SELECT ml_generate_embedding_result FROM ML.GENERATE_EMBEDDING(MODEL `{T}.mdl_embedding`,
           (SELECT '{question}' AS content), STRUCT(TRUE AS flatten_json_output))),
        top_k => 5))
    SELECT AI.GENERATE(CONCAT('Answer using only this context. Context: ', STRING_AGG(content, ' '),
                              ' Question: {question}'), endpoint => 'gemini-2.5-flash').result AS answer
    FROM hits""").to_dataframe()
    display(df_rag)
else:
    print("Skipped: RUN_BILLING_CELLS is False (Gemini and embeddings need billing).")

---
## Phase 3B: Data Governance, Quality & Security
*(JD Section 4: metadata management, data-quality checks, lineage, IAM, validation and monitoring)*

Executed and validated on `driiiportfolio.supply_chain_optimization`: 13 of 13 checks pass. Results are written to `dq_check_results` for monitoring.

In [ ]:
T = f"{PROJECT_ID}.{DATASET_ID}"

dq_sql = f"""
CREATE OR REPLACE TABLE `{T}.dq_check_results` AS
WITH s AS (SELECT * FROM `{T}.stg_inventory_transactions`),
     l AS (SELECT * FROM `{T}.stg_warehouse_logs`),
     d AS (SELECT warehouse_id FROM `{T}.dim_warehouses`),
checks AS (
  SELECT 'completeness' AS dimension, 'stg_inventory_transactions row count = source' AS check_name,
         (SELECT COUNT(*) FROM `{T}.fact_inventory_transactions`) - (SELECT COUNT(*) FROM s) AS failures
  UNION ALL SELECT 'completeness', 'stg_warehouse_logs row count = source',
         (SELECT COUNT(*) FROM `{T}.fact_warehouse_logs`) - (SELECT COUNT(*) FROM l)
  UNION ALL SELECT 'completeness', 'null keys in staging (transaction_id, log_id)',
         (SELECT COUNTIF(transaction_id IS NULL) FROM s) + (SELECT COUNTIF(log_id IS NULL) FROM l)
  UNION ALL SELECT 'uniqueness', 'duplicate transaction_id', (SELECT COUNT(*) - COUNT(DISTINCT transaction_id) FROM s)
  UNION ALL SELECT 'uniqueness', 'duplicate log_id', (SELECT COUNT(*) - COUNT(DISTINCT log_id) FROM l)
  UNION ALL SELECT 'referential integrity', 'transactions.warehouse_id missing from dim_warehouses',
         (SELECT COUNTIF(warehouse_id NOT IN (SELECT warehouse_id FROM d)) FROM s)
  UNION ALL SELECT 'referential integrity', 'logs.warehouse_id missing from dim_warehouses',
         (SELECT COUNTIF(warehouse_id NOT IN (SELECT warehouse_id FROM d)) FROM l)
  UNION ALL SELECT 'validity', 'transaction_type outside RESTOCK/OUTBOUND/ADJUSTMENT',
         (SELECT COUNTIF(transaction_type NOT IN ('RESTOCK','OUTBOUND','ADJUSTMENT')) FROM s)
  UNION ALL SELECT 'validity', 'stock_status outside IN_STOCK/LOW_STOCK/STOCKOUT',
         (SELECT COUNTIF(stock_status NOT IN ('IN_STOCK','LOW_STOCK','STOCKOUT')) FROM l)
  UNION ALL SELECT 'validity', 'unit_cost <= 0', (SELECT COUNTIF(unit_cost <= 0) FROM s)
  UNION ALL SELECT 'consistency', 'delivery before dispatch', (SELECT COUNTIF(delivery_timestamp < dispatch_timestamp) FROM l)
  UNION ALL SELECT 'consistency', 'actual_lead_time_days <> delivery - dispatch (days)',
         (SELECT COUNTIF(TIMESTAMP_DIFF(delivery_timestamp, dispatch_timestamp, DAY) <> actual_lead_time_days) FROM l)
  UNION ALL SELECT 'consistency', 'view totals <> staging totals (transactions, shipments)',
         ABS((SELECT SUM(total_transactions) FROM `{T}.vw_inventory_health`) - (SELECT COUNT(*) FROM s))
       + ABS((SELECT SUM(total_shipments) FROM `{T}.vw_warehouse_performance`) - (SELECT COUNT(*) FROM l))
  UNION ALL SELECT 'completeness', 'vw_shipment_detail row count <> staging shipment rows',
         ABS((SELECT COUNT(*) FROM `{T}.vw_shipment_detail`) - (SELECT COUNT(*) FROM l))
  UNION ALL SELECT 'consistency', 'vw_inventory_daily transaction total <> staging transactions',
         ABS((SELECT SUM(transactions) FROM `{T}.vw_inventory_daily`) - (SELECT COUNT(*) FROM s))
)
SELECT CURRENT_TIMESTAMP() AS run_at, dimension, check_name, failures,
       IF(failures = 0, 'PASS', 'FAIL') AS status
FROM checks
"""
execute_bq_query(client, dq_sql, "data-quality checks")

df_dq = client.query(f"SELECT dimension, check_name, failures, status FROM `{T}.dq_check_results` ORDER BY dimension, check_name").to_dataframe()
display(df_dq)
assert (df_dq["status"] == "PASS").all(), "Data-quality failures detected"
print(f"{(df_dq['status'] == 'PASS').sum()} of {len(df_dq)} checks passed")

In [ ]:
# Metadata management: table/column descriptions and labels (queryable in INFORMATION_SCHEMA)
metadata_sql = f"""
ALTER TABLE `{T}.stg_warehouse_logs` SET OPTIONS (
  description = 'Cleaned warehouse fulfillment logs. Source: fact_warehouse_logs. Grain: one row per shipment log.',
  labels = [('layer','staging'),('domain','supply_chain'),('pii','none')]);
ALTER TABLE `{T}.stg_warehouse_logs` ALTER COLUMN lead_time_variance_days SET OPTIONS (
  description = 'actual_lead_time_days minus planned_lead_time_days; > 0 means delayed');
ALTER TABLE `{T}.stg_inventory_transactions` SET OPTIONS (
  description = 'Cleaned inventory transactions. Source: fact_inventory_transactions. Grain: one row per transaction.',
  labels = [('layer','staging'),('domain','supply_chain'),('pii','none')]);
ALTER TABLE `{T}.stg_inventory_transactions` ALTER COLUMN total_transaction_value SET OPTIONS (
  description = 'quantity x unit_cost (NUMERIC, 2dp); adjustments can be negative');
ALTER VIEW `{T}.vw_warehouse_performance` SET OPTIONS (
  description = 'Shipment volume, delay and stockout KPIs by warehouse and carrier. Feeds Looker Studio.',
  labels = [('layer','reporting'),('domain','supply_chain')]);
ALTER VIEW `{T}.vw_shipment_detail` SET OPTIONS (description='Shipment-grain reporting view; primary Looker Studio source.',labels=[('layer','reporting'),('domain','supply_chain')]);
ALTER VIEW `{T}.vw_inventory_daily` SET OPTIONS (description='Daily inventory transaction summary; primary Looker Studio trend source.',labels=[('layer','reporting'),('domain','supply_chain')]);
ALTER VIEW `{T}.vw_inventory_health` SET OPTIONS (
  description = 'Inventory movement and valuation by warehouse and category. Feeds Looker Studio.',
  labels = [('layer','reporting'),('domain','supply_chain')]);
"""
execute_bq_query(client, metadata_sql, "metadata descriptions and labels")

# Lineage (derived from view definitions in INFORMATION_SCHEMA)
lineage = client.query(f"""
SELECT v.table_name AS view_name, ref AS upstream_table
FROM `{T}.INFORMATION_SCHEMA.VIEWS` v,
UNNEST(REGEXP_EXTRACT_ALL(v.view_definition, r'`[^`]+[.]{DATASET_ID}[.]([a-z_]+)`')) AS ref
GROUP BY 1, 2 ORDER BY 1, 2""").to_dataframe()
display(lineage)

```text
fact_inventory_transactions -> stg_inventory_transactions -> vw_inventory_health
                                                       -> vw_inventory_daily ------> Looker Studio trend

fact_warehouse_logs -> stg_warehouse_logs -> vw_warehouse_performance
                                      |--> vw_shipment_detail -------------------> Looker Studio KPIs/map/scatter/SKU
                                      |--> mdl_delay_classifier -> ml_model_metrics

dim_warehouses joins both semantic layers.
```

The live Looker Studio report is built outside the notebook; the repository contains the semantic-layer SQL and static reference build.

In [ ]:
# IAM review: who can access the dataset (emails masked for the portfolio)
ds = client.get_dataset(f"{PROJECT_ID}.{DATASET_ID}")

def mask(entity_type, entity_id):
    if entity_type == "userByEmail" and "@" in str(entity_id):
        name, domain = str(entity_id).split("@", 1)
        return f"{name[:1]}***@{domain}"
    return entity_id

df_iam = pd.DataFrame([
    {"role": e.role, "entity_type": e.entity_type, "entity": mask(e.entity_type, e.entity_id)}
    for e in ds.access_entries
])
display(df_iam)
print("Default table expiration (ms):", ds.default_table_expiration_ms, "| location:", ds.location)

### Least-privilege access model (design)

| Principal | Role | Scope | Why |
|---|---|---|---|
| Looker Studio viewers | `roles/bigquery.dataViewer` | `vw_*` views only (authorized views) | Dashboards read reporting views, never raw tables |
| Pipeline service account | `roles/bigquery.dataEditor` + `roles/bigquery.jobUser` | `supply_chain_optimization` dataset / project | Loads and transformations only |
| Analysts | `roles/bigquery.dataViewer` | `stg_*` and `vw_*` | Read-only exploration |
| Admins | `roles/bigquery.dataOwner` | Dataset | Schema and IAM changes, audited |

**Dataplex data-quality scan (not executed; needs Dataplex API and billing).** Verify against current documentation before use:

```bash
gcloud dataplex datascans create data-quality supply-chain-dq --location=us-central1 \
  --data-source-resource="//bigquery.googleapis.com/projects/driiiportfolio/datasets/supply_chain_optimization/tables/stg_warehouse_logs" \
  --data-quality-spec-file=dq_spec.yaml
```

The SQL checks above map one-to-one to Dataplex rules (`nonNullExpectation`, `uniquenessExpectation`, `setExpectation`, `rangeExpectation`).

---
## Phase 2C: Ingestion Patterns, Batch and Streaming (billing required)
*(JD Section 1: pipelines and integrations with Cloud Storage and Pub/Sub; batch and streaming ingestion)*

> **Status: code provided, NOT executed.** Cloud Storage and Pub/Sub need a billing-enabled project. Guarded by `RUN_BILLING_CELLS`.

- **Batch:** CSVs are landed in Cloud Storage and loaded with `load_table_from_uri` (replaces the local-file load in Phase 2).
- **Streaming:** synthetic shipment events are published to Pub/Sub, then a micro-batch consumer pulls them and appends to `stream_warehouse_logs` with a load job (streaming inserts are blocked in the sandbox).
- **Not implemented:** Dataflow and Dataproc. In production, Dataflow's Pub/Sub-to-BigQuery template would replace the consumer, and Dataproc would only be needed for existing Spark workloads.

In [ ]:
import json, random, uuid
import numpy as np
from datetime import datetime, timedelta

if RUN_BILLING_CELLS:
    from google.cloud import storage, pubsub_v1
    from google.api_core.exceptions import AlreadyExists

    # ---- Batch: land in Cloud Storage, then load from GCS ----
    BUCKET = f"{PROJECT_ID}-supply-chain-landing"
    gcs = storage.Client(project=PROJECT_ID)
    bucket = gcs.bucket(BUCKET)
    if not bucket.exists():
        bucket = gcs.create_bucket(BUCKET, location="US")

    for name in ["dim_warehouses", "fact_inventory_transactions", "fact_warehouse_logs"]:
        bucket.blob(f"raw/{name}.csv").upload_from_filename(f"data_export/{name}.csv")
        job = client.load_table_from_uri(
            f"gs://{BUCKET}/raw/{name}.csv", f"{PROJECT_ID}.{DATASET_ID}.{name}",
            job_config=bigquery.LoadJobConfig(source_format=bigquery.SourceFormat.CSV, skip_leading_rows=1,
                                              autodetect=True, write_disposition="WRITE_TRUNCATE"))
        job.result()
        print(f"[BATCH] gs://{BUCKET}/raw/{name}.csv -> {name}")

    # ---- Streaming: Pub/Sub publish, micro-batch pull, append to BigQuery ----
    publisher = pubsub_v1.PublisherClient()
    subscriber = pubsub_v1.SubscriberClient()
    topic_path = publisher.topic_path(PROJECT_ID, "warehouse-log-events")
    sub_path = subscriber.subscription_path(PROJECT_ID, "warehouse-log-events-sub")
    try:
        publisher.create_topic(request={"name": topic_path})
    except AlreadyExists:
        pass
    try:
        subscriber.create_subscription(request={"name": sub_path, "topic": topic_path})
    except AlreadyExists:
        pass

    now = datetime.utcnow()
    for _ in range(25):
        planned = random.randint(2, 7)
        actual = planned + int(np.random.choice([0,1,2,3,5],p=[0.50,0.25,0.15,0.07,0.03]))
        event = {"log_id": f"LOG-{uuid.uuid4().hex[:8].upper()}",
                 "warehouse_id": f"WH-{100 + random.randint(0, 4)}", "sku": f"SKU-{2000 + random.randint(0, 19)}",
                 "carrier": random.choice(["FedEx Freight", "UPS Supply Chain", "DHL Express", "JB Hunt"]),
                 "planned_lead_time_days": planned, "actual_lead_time_days": actual,
                 "stock_status": str(np.random.choice(["IN_STOCK","LOW_STOCK","STOCKOUT"],p=[0.70,0.20,0.10])),
                 "dispatch_timestamp": now.strftime("%Y-%m-%d %H:%M:%S"),
                 "delivery_timestamp": (now + timedelta(days=actual)).strftime("%Y-%m-%d %H:%M:%S")}
        publisher.publish(topic_path, json.dumps(event).encode("utf-8")).result()

    resp = subscriber.pull(request={"subscription": sub_path, "max_messages": 25}, timeout=30)
    rows = [json.loads(m.message.data.decode("utf-8")) for m in resp.received_messages]
    if rows:
        client.load_table_from_json(
            rows, f"{PROJECT_ID}.{DATASET_ID}.stream_warehouse_logs",
            job_config=bigquery.LoadJobConfig(autodetect=True, write_disposition="WRITE_APPEND")).result()
        subscriber.acknowledge(request={"subscription": sub_path, "ack_ids": [m.ack_id for m in resp.received_messages]})
    print(f"[STREAM] consumed and loaded {len(rows)} events")
else:
    print("Skipped: RUN_BILLING_CELLS is False (Cloud Storage and Pub/Sub need billing).")

---
## Phase 3C: Performance Tuning & Query Optimization
*(JD Section 1: data modeling, performance tuning and query optimization in BigQuery)*

Bytes scanned are measured with dry runs. **Column pruning** produces a large, measurable reduction (measured 2026-09-30: 219,559 bytes for `SELECT *` vs 79,559 for three columns, a 64% cut). **Clustering** shows no measurable difference at 2,500 rows: BigQuery bills a 10 MB minimum per table, and clustering pays off on tables in the gigabyte range. The partitioning failure in the delivery log is the other tuning lesson: partitioning must match the data's date range and the environment's expiration policy.

In [ ]:
def dry_run_bytes(sql: str) -> int:
    cfg = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    return client.query(sql, job_config=cfg).total_bytes_processed

T = f"{PROJECT_ID}.{DATASET_ID}"
tests = {
    "SELECT * (all columns)": f"SELECT * FROM `{T}.fact_warehouse_logs`",
    "3 needed columns only": f"SELECT warehouse_id, carrier, stock_status FROM `{T}.fact_warehouse_logs`",
    "Filter on unclustered raw table": f"SELECT warehouse_id, carrier, COUNT(*) n FROM `{T}.fact_warehouse_logs` WHERE warehouse_id = 'WH-100' GROUP BY 1, 2",
    "Same filter on clustered staging table": f"SELECT warehouse_id, carrier, COUNT(*) n FROM `{T}.stg_warehouse_logs` WHERE warehouse_id = 'WH-100' GROUP BY 1, 2",
}
df_perf = pd.DataFrame([{"query": k, "bytes_processed": dry_run_bytes(v)} for k, v in tests.items()])
full = df_perf.loc[0, "bytes_processed"]
df_perf["vs_select_star"] = (df_perf["bytes_processed"] / full).map("{:.0%}".format)
display(df_perf)

---
## Continuous Learning & Reusable Components
*(JD Section 5)*

**Reusable components in this repository:** the parameterized synthetic-data generator, the staging and view SQL, the data-quality check suite (`dq_check_results`), the model train, evaluate and gate cells, and the dry-run cost benchmark. All read `PROJECT_ID` and `DATASET_ID`, so they can be reused on another dataset.

**Current GCP features used:** BigQuery ML (`CREATE MODEL`, `ML.EVALUATE`), `INFORMATION_SCHEMA` for lineage, ALTER-based metadata management, and (billing-gated) `AI.GENERATE`, `ML.GENERATE_EMBEDDING` and `VECTOR_SEARCH`.

**Release tracking:** BigQuery release notes (cloud.google.com/bigquery/docs/release-notes), Google Cloud release notes (cloud.google.com/release-notes), Vertex AI release notes (cloud.google.com/vertex-ai/docs/release-notes). Re-check syntax in the billing-gated cells against these before running.

## Job Description Mapping: Google Data Specialist (R00336594)

**Legend:** ✅ executed/validated · 🟡 code/design provided but not executed in the no-billing environment · ⬜ not covered.

| Capability | Evidence | Status |
|---|---|---|
| ETL/ELT + BigQuery | Python generator, load jobs, staging and semantic views | ✅ |
| Cloud Storage / Pub/Sub | Billing-gated production patterns | 🟡 |
| Dataflow / Dataproc | Production design paths documented | 🟡 |
| BigQuery data modeling | Raw → staging → semantic views | ✅ |
| Query optimization | Dry-run bytes, column pruning, clustering benchmark | ✅ |
| Looker Studio | Two-source semantic layer + reference build | 🟡 live report remains external |
| BigQuery ML | Logistic regression, evaluation and AUC gate | ✅ |
| MLOps | `ml_model_metrics` run history and promotion decision | ✅ |
| Gemini / Vertex AI | Billing-gated code; not executed | 🟡 |
| Data quality | 15-check `dq_check_results` suite | ✅ |
| Metadata / lineage | `ALTER ... SET OPTIONS` + `INFORMATION_SCHEMA` | ✅ |
| IAM/security | Access review + least-privilege design | ✅ / design |
| Requirements traceability | Simulated brief and delivery log | ✅ |

The repository does not claim execution of Cloud Storage, Pub/Sub, Dataflow, Dataproc, Dataplex or Vertex AI managed services in the no-billing environment.

## Phase 4D: Looker Studio Control Tower — Final Build Specification

**Status:** Final dashboard specification synchronized to the completed five dashboard corrections on 2026-09-30. The live Looker Studio report is external to this repository.

### Data sources
- BigQuery project: `driiiportfolio`
- Dataset: `supply_chain_optimization`
- `vw_shipment_detail` → **Shipments**
- `vw_inventory_daily` → **Inventory Daily**
- Use Owner's Credentials when appropriate for a portfolio report.

### Date configuration
- Default: **Custom — Jan 1, 2026 through Jun 30, 2026**
- Shipments date range dimension: `dispatch_date`
- Inventory date range dimension: `transaction_date`
- Do not use Last 90 Days for validation.
- Report controls: `warehouse_id`, `carrier`, and date range.

### Page 1 — Executive Control Tower

| Component | Source | Configuration | Expected |
|---|---|---|---|
| Fulfillment Rate | Shipments | `SUM(is_on_time) / COUNT(log_id)` | **52.1%** |
| Avg Lead-Time Variance | Shipments | `AVG(lead_time_variance_days)` | **+0.87 days** |
| Stockout Incidents | Shipments | `SUM(is_stockout)` | **247** |
| Low-Stock Warnings | Shipments | `SUM(is_low_stock)` | **444** |
| Gross Transaction Value | Inventory Daily | `SUM(gross_value)` | **$44,350,562.38** |
| Total Shipments | Shipments | `COUNT(log_id)` | **2,500** |
| Distribution map | Shipments | city; size=`SUM(is_stockout)`; color=`AVG(lead_time_variance_days)` | Reconcile to BQ |
| Category value trend | Inventory Daily | transaction_date; breakdown=category; `SUM(gross_value)` | **$44,350,562.38** total |

**Map caption:** “Warehouse locations are sized by stockout incidents and colored by average lead-time variance to highlight where inventory availability and fulfillment timing create the greatest operational risk.”

**Trend configuration:** Y-axis title = `Gross Transaction Value (USD)`; minimum = `0`; sort `transaction_date` ascending.

**Trend caption:** “Daily gross transaction value is segmented by inventory category to show how the composition and magnitude of warehouse activity change throughout the reporting period.”

### Page 2 — Shipment Performance & Risk

**Scatter/bubble:** X=`AVG(planned_lead_time_days)`; Y=`AVG(actual_lead_time_days)`; size=`SUM(is_delayed)/COUNT(log_id)`; dimensions=`carrier`, `planned_lead_time_days`; color=`carrier`; sort by planned lead time ascending, then carrier ascending.

**Scatter caption:** “Carrier-level bubbles compare planned versus actual lead time, with larger bubbles representing higher delay rates so that slower and less reliable shipment groupings are immediately visible.”

**SKU table:** dimension=`sku`; metrics=`COUNT(log_id)`, `SUM(is_stockout)`, Stockout Rate, `AVG(lead_time_variance_days)`; primary sort = **Stockout Rate descending**; secondary sort = **Stockout Incidents descending**. Use the unrounded Stockout Rate calculation for sorting. Do not add category: shipment logs do not contain category.

**Table caption:** “The SKU ranking identifies products with the highest stockout exposure while retaining shipment volume and lead-time variance to distinguish concentrated inventory risk from low-volume anomalies.”

### Calculated fields

- `Fulfillment Rate = SUM(is_on_time) / COUNT(log_id)`
- `Delay Rate = SUM(is_delayed) / COUNT(log_id)`
- `Stockout Rate = SUM(is_stockout) / COUNT(log_id)`
- `Avg Lead-Time Variance = AVG(lead_time_variance_days)`

These use precomputed 0/1 flags instead of SQL `COUNTIF`.

### Executive-facing presentation labels

- `is_stockout` → **Stockout Incidents**
- `lead_time_variance_days` → **Avg Lead-Time Variance (days)** when displayed as an aggregate
- `gross_value` → **Gross Transaction Value (USD)**

### Acceptance values

**2,500 shipments · 1,303 on-time/early · 1,197 delayed · 247 stockouts · 444 low-stock · 52.1% fulfillment · +0.87 days · 1,500 inventory transactions · $44,350,562.38 gross value · 15/15 DQ checks.**

Any mismatch means the source, date range, aggregation, filter, or chart configuration must be corrected before publication.


## Repository handoff

The repository files are the canonical upload set. The notebook is the end-to-end execution artifact; SQL, Python and documentation are separated under `scripts/` and `docs/`.

## Final handoff checklist

1. Run notebook top-to-bottom.
2. Confirm 15/15 DQ PASS.
3. Confirm 2,500 shipment and 1,500 transaction dashboard-source reconciliation.
4. Build/maintain Looker Studio from the two current semantic views.
5. Validate the fixed Jan 1–Jun 30, 2026 range.
6. Confirm the six Page 1 scorecards, including **Total Shipments = 2,500** as a separate metric from Gross Transaction Value.
7. Confirm Avg Lead-Time Variance displays **+0.87 days**.
8. Confirm the SKU table sorts **Stockout Rate DESC**, then **Stockout Incidents DESC**.
9. Confirm executive-facing labels and the inventory trend Y-axis minimum of **0**.
10. Compare the live report to the acceptance values before publishing its link.
